In [ ]:
import os
import numpy as np
import pickle
import torch
from argparse import ArgumentParser
from tqdm import tqdm
import glob

from articulate.model import ParametricModel
from articulate import math
from config import paths, datasets


In [ ]:
import os
import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch import seed_everything
from argparse import ArgumentParser
from pathlib import Path
from typing import List
from tqdm import tqdm 
import wandb

from constants import MODULES
from data import PoseDataModule
from utils.file_utils import (
    get_datestring, 
    make_dir, 
    get_dir_number, 
    get_best_checkpoint
)
from config import paths, train_hypers, finetune_hypers




In [ ]:
type(train_hypers)

In [ ]:
train_hypers.num_epochs

In [ ]:

# set precision for Tensor cores
torch.set_float32_matmul_precision('medium')


class TrainingManager:
    """Manage training of MobilePoser modules."""
    def __init__(self, finetune: str=None, fast_dev_run: bool=False):
        self.finetune = finetune
        self.fast_dev_run = fast_dev_run
        self.hypers = finetune_hypers if finetune else train_hypers

    def _setup_wandb_logger(self, save_path: Path):
        wandb_logger = WandbLogger(
            project=save_path.name, 
            name=get_datestring(),
            save_dir=save_path
        ) 
        return wandb_logger

    def _setup_callbacks(self, save_path):
        checkpoint_callback = ModelCheckpoint(
                monitor="validation_step_loss",
                save_top_k=3,
                mode="min",
                verbose=False,
                dirpath=save_path, 
                save_weights_only=True,
                filename="{epoch}-{validation_step_loss:.4f}"
                )
        return checkpoint_callback

    def _setup_trainer(self, module_path: Path):
        print("Module Path: ", module_path.name, module_path)
        logger = self._setup_wandb_logger(module_path) 
        checkpoint_callback = self._setup_callbacks(module_path)
        trainer = L.Trainer(
                fast_dev_run=self.fast_dev_run,
                min_epochs=self.hypers.num_epochs,
                max_epochs=self.hypers.num_epochs,
                devices=[self.hypers.device], 
                accelerator=self.hypers.accelerator,
                logger=logger,
                callbacks=[checkpoint_callback],
                deterministic=True
                )
        return trainer

    def train_module(self, model: L.LightningModule, module_name: str, checkpoint_path: Path):
        # set the appropriate hyperparameters
        model.hypers = self.hypers 

        # create directory for module
        module_path = checkpoint_path / module_name
        make_dir(module_path)
        datamodule = PoseDataModule(finetune=self.finetune)
        trainer = self._setup_trainer(module_path)

        print()
        print("-" * 50)
        print(f"Training Module: {module_name}")
        print("-" * 50)
        print()

        try:
            trainer.fit(model, datamodule=datamodule)
        finally:
            wandb.finish()
            del model
            torch.cuda.empty_cache()


def get_checkpoint_path(finetune: str, init_from: str):
    if finetune:
        # finetune from a checkpoint
        parts = init_from.split(os.path.sep)
        checkpoint_path = Path(parts[0]) / parts[1]
        finetune_dir = f"finetuned_{finetune}"
        checkpoint_path = checkpoint_path / finetune_dir
    else:
        # make directory for trained models
        dir_name = get_dir_number(paths.checkpoint)
        checkpoint_path = paths.checkpoint / str(dir_name)

    # create directory if it does not exist
    checkpoint_path.mkdir(parents=True, exist_ok=True)

    return checkpoint_path


In [ ]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = None
init_from =  "scratch"
fast_dev_run = False

module = 'poser'
checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

In [ ]:
if module:
    if module not in MODULES:
        raise ValueError(f"Module {module} not found.")

    module_name = module              # "poser"
    module_cls = MODULES[module_name] # models.poser.Poser (a class)
    
    model_dir = Path(init_from)
    model = module_cls()              # init model from scratch
    print('model: ',model)
    if finetune:
        model_path = get_best_checkpoint(model_dir)
        model = module_cls.from_pretrained(
            model_path=os.path.join(model_dir, model_path)
        )

    training_manager.train_module(model, module_name, checkpoint_path)
else:
    # train all modules
    for module_name, module_cls in MODULES.items():
        training_manager.train_module(module_cls(), module_name, checkpoint_path)


In [ ]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = None
init_from =  "scratch"
fast_dev_run = False

module = 'joints'
checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

In [ ]:
if module:
    if module not in MODULES:
        raise ValueError(f"Module {module} not found.")

    module_name = module              # "poser"
    module_cls = MODULES[module_name] # models.poser.Poser (a class)

    model_dir = Path(init_from)
    model = module_cls()              # init model from scratch

    if finetune:
        model_path = get_best_checkpoint(model_dir)
        model = module_cls.from_pretrained(
            model_path=os.path.join(model_dir, model_path)
        )

    training_manager.train_module(model, module_name, checkpoint_path)
else:
    # train all modules
    for module_name, module_cls in MODULES.items():
        training_manager.train_module(module_cls(), module_name, checkpoint_path)


In [ ]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = None
init_from =  "scratch"
fast_dev_run = False

module = 'poser'
checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

In [ ]:
if module:
    if module not in MODULES:
        raise ValueError(f"Module {module} not found.")

    module_name = module              # "poser"
    module_cls = MODULES[module_name] # models.poser.Poser (a class)

    model_dir = Path(init_from)
    model = module_cls()              # init model from scratch

    if finetune:
        model_path = get_best_checkpoint(model_dir)
        model = module_cls.from_pretrained(
            model_path=os.path.join(model_dir, model_path)
        )

    training_manager.train_module(model, module_name, checkpoint_path)
else:
    # train all modules
    for module_name, module_cls in MODULES.items():
        training_manager.train_module(module_cls(), module_name, checkpoint_path)


In [ ]:


if __name__ == "__main__":
    parser = ArgumentParser()
    parser.add_argument("--module", default=None)
    parser.add_argument("--fast-dev-run", action="store_true")
    parser.add_argument("--finetune", type=str, default=None)
    parser.add_argument("--init-from", nargs="?", default="scratch", type=str)
    args = parser.parse_args()

    # set seed for reproducible results
    
    # create checkpoint directory, if missing
    

    # initialize training manager
    

    # train single module
    if module:
        if module not in MODULES.keys():
            raise ValueError(f"Module {module} not found.")

        model_dir = Path(init_from)
        module = MODULES[module]
        model = module() # init model from scratch

        if finetune: 
            model_path = get_best_checkpoint(model_dir)
            model = module.from_pretrained(model_path=os.path.join(model_dir, model_path)) # load pre-trained model

        training_manager.train_module(model, module, checkpoint_path)
    else:
        # train all modules
        for module_name, module in MODULES.items():
            training_manager.train_module(module(), module_name, checkpoint_path)

In [ ]:
# python train.py --module joints --init-from checkpoints/$2/joints --finetune dip
#     python train.py --module poser --init-from checkpoints/$2/poser --finetune dip

In [ ]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = "dip"
init_from =  r"C:\deepika\2.mobileposer\MobilePoser\mobileposer\checkpoints\8\joints"
fast_dev_run = False

module = 'joints'
checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

if module:
    if module not in MODULES:
        raise ValueError(f"Module {module} not found.")

    module_name = module              # "poser"
    module_cls = MODULES[module_name] # models.poser.Poser (a class)

    model_dir = Path(init_from)
    model = module_cls()              # init model from scratch

    if finetune:
        print('finetuning sista')
        model_path = get_best_checkpoint(model_dir)
        model = module_cls.from_pretrained(
            model_path=os.path.join(model_dir, model_path)
        )

    training_manager.train_module(model, module_name, checkpoint_path)
else:
    # train all modules
    for module_name, module_cls in MODULES.items():
        training_manager.train_module(module_cls(), module_name, checkpoint_path)


In [ ]:
"""
Combine network weights into a single weight file. 
"""

import os
import re
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm 
from argparse import ArgumentParser

from models import MobilePoserNet, Poser, Joints, Velocity, FootContact
from constants import MODULES
from utils.file_utils import get_file_number


def load_module_weights(module_name, weight_path):
    try:
        return MODULES[module_name].load_from_checkpoint(weight_path)
    except Exception as e:
        print(f"Error loading {module_name} weights from {weight_path}: {e}")
        return None


from pathlib import Path

def get_module_path(module_name, checkpoint_dir, finetune=None):
    module_path = Path(checkpoint_dir)  # <-- use it directly
    if finetune and module_name in ["poser", "joints"]:
        module_path = module_path / f"finetuned_{finetune}" / module_name
    else:
        module_path = module_path / module_name
    return module_path

checkpoint_dir = r"C:\deepika\2.mobileposer\MobilePoser\mobileposer\deepika"
finetune = "dip"

In [ ]:
# checkpoint_dir =r"C:\deepika\2.mobileposer\MobilePoser\mobileposer\deepika"
# finetune = "dip"
import os
import re
from pathlib import Path

def get_best_checkpoint(path):
    path = Path(path)
    if not path.exists() or not path.is_dir():
        print(f"[skip] checkpoint dir not found: {path}")
        return None

    pattern = re.compile(r"epoch=\d+-validation_step_loss=([0-9.]+)\.ckpt$")
    files = [f.name for f in path.iterdir() if f.is_file() and pattern.search(f.name)]
    if not files:
        return None

    return min(files, key=lambda x: float(pattern.search(x).group(1)))


In [ ]:
checkpoints = {}
for module_name in MODULES.keys():
    module_path = get_module_path(module_name, checkpoint_dir, finetune)
    best_ckpt = get_best_checkpoint(module_path)
    if best_ckpt:
        checkpoints[module_name] = load_module_weights(module_name, module_path / best_ckpt)
        print(f"Module: {module_name.ljust(15)} | Best Checkpoint: {best_ckpt}")
    else:
        print(f"No checkpoint found for {module_name} in {module_path}")

model_name = "base_model.pth" if not finetune else "model_finetuned.pth"
model = MobilePoserNet(**checkpoints)

model_path = Path(checkpoint_dir) / model_name  # <-- write into that folder
torch.save(model.state_dict(), model_path)
print(f"Model written to {model_path}.")

In [ ]:
import os
import numpy as np
import torch
from argparse import ArgumentParser
import tqdm 

from config import *
from helpers import * 
import articulate as art
from constants import MODULES
from utils.model_utils import load_model
from data import PoseDataset
from models import MobilePoserNet


class PoseEvaluator:
    def __init__(self):
        self._eval_fn = art.FullMotionEvaluator(paths.smpl_file, joint_mask=torch.tensor([2, 5, 16, 20]), fps=datasets.fps)

    def eval(self, pose_p, pose_t, joint_p=None, tran_p=None, tran_t=None):
        pose_p = pose_p.clone().view(-1, 24, 3, 3)
        pose_t = pose_t.clone().view(-1, 24, 3, 3)
        tran_p = tran_p.clone().view(-1, 3)
        tran_t = tran_t.clone().view(-1, 3)
        pose_p[:, joint_set.ignored] = torch.eye(3, device=pose_p.device)
        pose_t[:, joint_set.ignored] = torch.eye(3, device=pose_t.device)

        errs = self._eval_fn(pose_p, pose_t, tran_p=tran_p, tran_t=tran_t)
        return torch.stack([errs[9], errs[3], errs[9], errs[0]*100, errs[7]*100, errs[1]*100, errs[4] / 100, errs[6]])

    @staticmethod
    def print(errors):
        for i, name in enumerate(['SIP Error (deg)', 'Angular Error (deg)', 'Masked Angular Error (deg)',
                                  'Positional Error (cm)', 'Masked Positional Error (cm)', 'Mesh Error (cm)', 
                                  'Jitter Error (100m/s^3)', 'Distance Error (cm)']):
            print('%s: %.2f (+/- %.2f)' % (name, errors[i, 0], errors[i, 1]))


@torch.no_grad()
def evaluate_pose(model, dataset, num_past_frame=20, num_future_frame=5, evaluate_tran=False):
    # specify device
    device = model_config.device

    # load data
    xs, ys = zip(*[(imu.to(device), (pose.to(device), tran)) for imu, pose, joint, tran in dataset])

    # setup Pose Evaluator
    evaluator = PoseEvaluator()

    # track errors
    offline_errs, online_errs = [], []
    tran_errors = {window_size: [] for window_size in list(range(1, 8))}
    
    model.eval()
    with torch.no_grad():
        for x, y in tqdm.tqdm(list(zip(xs, ys))):
            model.reset()
            pose_p_offline, joint_p_offline, tran_p_offline, _ = model.forward_offline(x.unsqueeze(0), [x.shape[0]])
            pose_t, tran_t = y
            pose_t = art.math.r6d_to_rotation_matrix(pose_t)

            if getenv("ONLINE"):
                online_results = [model.forward_online(f) for f in torch.cat((x, x[-1].repeat(num_future_frame, 1)))]
                pose_p_online, joint_p_online, tran_p_online, _ = [torch.stack(_)[num_future_frame:] for _ in zip(*online_results)]

            if evaluate_tran:
                # compute gt move distance at every frame 
                move_distance_t = torch.zeros(tran_t.shape[0])
                v = (tran_t[1:] - tran_t[:-1]).norm(dim=1)
                for j in range(len(v)):
                    move_distance_t[j + 1] = move_distance_t[j] + v[j] # distance travelled

                for window_size in tran_errors.keys():
                    # find all pairs of start/end frames where gt moves `window_size` meters
                    frame_pairs = []
                    start, end = 0, 1
                    while end < len(move_distance_t):
                        if move_distance_t[end] - move_distance_t[start] < window_size: # if not less than the window_size (in meters)
                            end += 1
                        else:
                            if len(frame_pairs) == 0 or frame_pairs[-1][1] != end:
                                frame_pairs.append((start, end))
                            start += 1

                    # calculate mean distance error 
                    errs = []
                    for start, end in frame_pairs:
                        vel_p = tran_p_offline[end] - tran_p_offline[start]
                        vel_t = (tran_t[end] - tran_t[start]).to(device)
                        errs.append((vel_t - vel_p).norm() / (move_distance_t[end] - move_distance_t[start]) * window_size)
                    if len(errs) > 0:
                        tran_errors[window_size].append(sum(errs) / len(errs))

            offline_errs.append(evaluator.eval(pose_p_offline, pose_t, tran_p=tran_p_offline, tran_t=tran_t))
            if getenv("ONLINE"):
                online_errs.append(evaluator.eval(pose_p_online, pose_t, tran_p=tran_p_online, tran_t=tran_t))

    # print joint errors
    print('============== offline ================')
    evaluator.print(torch.stack(offline_errs).mean(dim=0))
    if getenv("ONLINE"):
        print('============== online ================')
        evaluator.print(torch.stack(online_errs).mean(dim=0))
    
    # print translation errors
    if evaluate_tran:
        print([0] + [torch.tensor(_).mean() for _ in tran_errors.values()])

model = r"C:\deepika\2.mobileposer\MobilePoser\mobileposer\deepika\model_finetuned.pth"
dataset = "dip"
model = load_model(model)

# load dataset
if dataset not in datasets.test_datasets:
    raise ValueError(f"Test dataset: {dataset} not found.")
dataset = PoseDataset(fold='test', evaluate=dataset)

# evaluate pose
print(f"Starting evaluation: {dataset.capitalize()}")
evaluate_pose(model, dataset)

In [ ]:
model_path = r"C:\deepika\2.mobileposer\MobilePoser\mobileposer\deepika\model_finetuned.pth"
dataset_name = "dip"

model = load_model(model_path)

if dataset_name not in datasets.test_datasets:
    raise ValueError(f"Test dataset: {dataset_name} not found.")

dataset = PoseDataset(fold="test", evaluate=dataset_name)

print(f"Starting evaluation: {dataset_name.capitalize()}")
evaluate_pose(model, dataset)
